# Experimento 4.4: las dos formas de guardar los k-mers

- colección separada: un documento por secuencia y por k.
- embebida: un documento por secuencia con un arreglo kmers que trae los 6 perfiles.

In [1]:
import time
from pymongo import MongoClient

db= MongoClient("mongodb://localhost:27017")["panteon_prueba"]
print(db.kmers.count_documents({}))

0


## Tiempo de carga

In [2]:
origen= list(db.kmers.find({},{"_id":0}))

db.kmers_a.drop()
inicio= time.perf_counter()
db.kmers_a.insert_many(origen)
tiempo_a= time.perf_counter() - inicio

db.kmers_b.drop()
inicio= time.perf_counter()
por_secuencia= {}
for d in origen:
    sid= d["secuencia_id"]
    if sid not in por_secuencia:
        por_secuencia[sid]= {"secuencia_id":sid,"te_orden":d["te_orden"],"kmers":[]}
    por_secuencia[sid]["kmers"].append({"k":d["k"],"total_ventanas":d["total_ventanas"],"conteos":d["conteos"]})
db.kmers_b.insert_many(list(por_secuencia.values()))
tiempo_b= time.perf_counter() - inicio

print("A:",round(tiempo_a,2),"s")
print("B:",round(tiempo_b,2),"s")
print(db.kmers_a.count_documents({}),db.kmers_b.count_documents({}))

TypeError: documents must be a non-empty list

In [ ]:
sid= db.kmers_a.find_one()["secuencia_id"]
orden= db.kmers_a.find_one()["te_orden"]
cond= {"k":6,"total_ventanas":{"$gt":1000}}

consultas_a= [
    ({"secuencia_id":sid,"k":6},None),
    ({"te_orden":orden,"k":2},None),
    (cond,None)
]
consultas_b= [
    ({"secuencia_id":sid},{"kmers":{"$elemMatch":{"k":6}}}),
    ({"te_orden":orden},{"kmers":{"$elemMatch":{"k":2}}}),
    ({"kmers":{"$elemMatch":cond}},{"kmers":{"$elemMatch":cond}})
]
resultados= []

def medir(coleccion,filtro,proyeccion,con_indice):
    tiempos= []
    for i in range(3):
        cursor= coleccion.find(filtro,proyeccion)
        if not con_indice:
            cursor= cursor.hint([("$natural",1)])
        datos= cursor.explain()["executionStats"]
        tiempos.append(datos["executionTimeMillis"])
    return min(tiempos),datos["totalDocsExamined"],datos["nReturned"]

def probar(con_indice):
    for i in range(3):
        filtro,proy= consultas_a[i]
        ms,docs,devueltos= medir(db.kmers_a,filtro,proy,con_indice)
        resultados.append(("A",i+1,"con" if con_indice else "sin",ms,docs,devueltos))
        filtro,proy= consultas_b[i]
        ms,docs,devueltos= medir(db.kmers_b,filtro,proy,con_indice)
        resultados.append(("B",i+1,"con" if con_indice else "sin",ms,docs,devueltos))

probar(False)

db.kmers_a.create_index([("secuencia_id",1),("k",1)])
db.kmers_a.create_index([("te_orden",1),("k",1)])
db.kmers_a.create_index([("k",1),("total_ventanas",1)])
db.kmers_b.create_index([("secuencia_id",1)])
db.kmers_b.create_index([("te_orden",1)])
db.kmers_b.create_index([("kmers.k",1),("kmers.total_ventanas",1)])

probar(True)

## Resultados

In [ ]:
escala= max(r[3] for r in resultados) or 1
print(f"{'estrategia':<11}{'consulta':<9}{'indice':<7}{'ms':>7}{'docs':>8}{'devueltos':>10}")
for r in resultados:
    print(f"{r[0]:<11}{r[1]:<9}{r[2]:<7}{r[3]:>7}{r[4]:>8}{r[5]:>10}  "+"█"*max(1,int(r[3]/escala*40)))

In [ ]:
for nombre in ["kmers_a","kmers_b"]:
    s= db.command("collStats",nombre)
    print(nombre,round(s["storageSize"]/1e6,1),"MB datos",round(s["totalIndexSize"]/1e6,1),"MB índices")